# Dispatch Price Source

In [3]:
from pathlib import Path
import traceback
import nemosis
import pandas as pd

Fetch and cache regional dispatch prices from NEMOSIS.

In [4]:


def _process_month(start: str, end: str) -> pd.DataFrame:
    REGION_COLUMNS = {"NSW1": "nsw_price", "QLD1": "qld_price", "VIC1": "vic_price", "SA1": "sa_price"}

    df = nemosis.dynamic_data_compiler(
        start_time=start,
        end_time=end,
        table_name="DISPATCHPRICE",
        raw_data_location=str(Path("Pre_processing/nemosis_raw_dispatch_price")),
        select_columns=["SETTLEMENTDATE", "REGIONID", "RRP"],
        filter_cols=["REGIONID"],
        filter_values=[list(REGION_COLUMNS.keys())],
        fformat="feather",
        keep_csv=False,
    )
    df["SETTLEMENTDATE"] = pd.to_datetime(df["SETTLEMENTDATE"])
    df["RRP"] = pd.to_numeric(df["RRP"], errors="coerce")
    return (
        df.groupby(["SETTLEMENTDATE", "REGIONID"])["RRP"]
        .mean()
        .unstack("REGIONID")
        .resample("5min").mean()
        .rename(columns=REGION_COLUMNS)
    )


def main():
    data_start = pd.Timestamp("2018/01/01")
    data_end   = pd.Timestamp("2026/01/01")

    processed_path = Path("Processed_data/1_dispatch_price.csv")
    already_processed = set()
    if processed_path.exists():
        existing_idx = pd.read_csv(processed_path, usecols=[0], parse_dates=True).iloc[:, 0]
        already_processed = set(pd.to_datetime(existing_idx).dt.to_period("M"))
        print(f"  Found {len(already_processed)} already-processed month(s) — will skip.", flush=True)

    total_months = pd.date_range(data_start, data_end - pd.Timedelta(days=1), freq="MS")
    monthly_dfs = []

    for i, month in enumerate(total_months):
        if month.to_period("M") in already_processed:
            print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} skipping (already processed).", flush=True)
            continue
        start = month.strftime("%Y/%m/%d %H:%M:%S")
        end   = (month + pd.offsets.MonthEnd(0) + pd.Timedelta(days=1)).strftime("%Y/%m/%d %H:%M:%S")
        print(f"{i + 1:3d}/{len(total_months)} {month:%Y-%m} fetching...", flush=True)
        try:
            monthly_dfs.append(_process_month(start, end))
            for f in Path("Pre_processing/nemosis_raw_dispatch_price").glob(f"*{month.strftime('%Y%m')}*.feather"):
                f.unlink()
            print(f"  {month:%Y-%m} raw files deleted.", flush=True)
        except Exception:
            print(f"  WARNING: {month:%Y-%m} failed:\n{traceback.format_exc()}", flush=True)

    if not monthly_dfs:
        print("No new months to process.", flush=True)
        return pd.read_csv(processed_path, index_col=0, parse_dates=True).tail()

    df = pd.concat(monthly_dfs).sort_index()
    df = df[~df.index.duplicated(keep="last")]

    if processed_path.exists():
        existing = pd.read_csv(processed_path, index_col=0, parse_dates=True)
        existing.index = pd.to_datetime(existing.index, format="mixed")
        df = pd.concat([existing, df])
        df = df[~df.index.duplicated(keep="last")].sort_index()

    df.index.name = "SETTLEMENTDATE"
    df.to_csv(processed_path)
    return df.tail()

main()


  Found 97 already-processed month(s) — will skip.
  1/96 2018-01 skipping (already processed).
  2/96 2018-02 skipping (already processed).
  3/96 2018-03 skipping (already processed).
  4/96 2018-04 skipping (already processed).
  5/96 2018-05 skipping (already processed).
  6/96 2018-06 skipping (already processed).
  7/96 2018-07 skipping (already processed).
  8/96 2018-08 skipping (already processed).
  9/96 2018-09 skipping (already processed).
 10/96 2018-10 skipping (already processed).
 11/96 2018-11 skipping (already processed).
 12/96 2018-12 skipping (already processed).
 13/96 2019-01 skipping (already processed).
 14/96 2019-02 skipping (already processed).
 15/96 2019-03 skipping (already processed).
 16/96 2019-04 skipping (already processed).
 17/96 2019-05 skipping (already processed).
 18/96 2019-06 skipping (already processed).
 19/96 2019-07 skipping (already processed).
 20/96 2019-08 skipping (already processed).
 21/96 2019-09 skipping (already processed).
 22/

,nsw_price,qld_price,sa_price,vic_price
SETTLEMENTDATE,,,,
2025-12-31 23:40:00,58.19478,63.83005,22.97707,8.95000
2025-12-31 23:45:00,68.25496,75.75000,26.13849,8.95000
2025-12-31 23:50:00,77.33239,84.81000,29.23525,8.95000
2025-12-31 23:55:00,67.93443,75.75000,18.20000,1.01000
2026-01-01 00:00:00,77.88970,84.81000,56.12367,64.45527
